In [2]:
from collections import deque
import numpy as np
import pandas as pd
from pathlib import Path

WINDOW_DAYS = 42
DECAY_LAMBDA = 0.99

INTERIM_DIR = Path("../data/interim")

df = pd.read_csv(INTERIM_DIR / "atp_all_matches_with_elo.csv")

df["tourney_date"] = pd.to_datetime(df["tourney_date"], format = "%Y%m%d")

In [4]:
def compute_recent_form(history: deque, current_date, current_surface, window_days, decay_lambda):

    if len(history) == 0:
        return np.nan, 0, np.nan
    
    weighted_wins = 0.0
    weighted_total = 0.0
    surface_count = 0
    total_count = 0

    for match_date, result, surface in history:
        days_ago = (current_date - match_date).days

        if days_ago > window_days:
            continue

        weight = decay_lambda ** days_ago

        weighted_wins += weight * result
        weighted_total += weight
        total_count += 1

        if surface == current_surface:
            surface_count += 1
    
    if weighted_total == 0:
        return np.nan, 0, np.nan
    
    recent_win_rate = weighted_wins /weighted_total
    recent_matches_played = total_count
    recent_surface_relevance = surface_count / total_count

    return recent_win_rate, recent_matches_played, recent_surface_relevance

In [7]:
player_history = {}

winner_recent_form_list = []
loser_recent_form_list = []
winner_recent_matches_list = []
loser_recent_matches_list = []
winner_surface_relevance_list = []
loser_surface_relevance_list = []


for _, row in df.iterrows():

    w_id = row["winner_id"]
    l_id = row["loser_id"]

    current_date = row["tourney_date"]
    surface = row["surface"]

    if w_id not in player_history:
        player_history[w_id] = deque()
    if l_id not in player_history:
        player_history[l_id] = deque()

    for pid in [w_id, l_id]:
        while player_history[pid] and (current_date - player_history[pid][0][0]).days > WINDOW_DAYS:
            player_history[pid].popleft()

    w_form, w_matches, w_surface_rel = compute_recent_form(
        player_history[w_id], current_date, surface, WINDOW_DAYS, DECAY_LAMBDA
    )

    l_form, l_matches, l_surface_rel = compute_recent_form(
        player_history[l_id], current_date, surface, WINDOW_DAYS, DECAY_LAMBDA
    )

    winner_recent_form_list.append(w_form)
    loser_recent_form_list.append(l_form)
    winner_recent_matches_list.append(w_matches)
    loser_recent_matches_list.append(l_matches)
    winner_surface_relevance_list.append(w_surface_rel)
    loser_surface_relevance_list.append(l_surface_rel)

    player_history[w_id].append((current_date, 1, surface))
    player_history[l_id].append((current_date, 0, surface))
    

df["winner_recent_form"] = winner_recent_form_list
df["loser_recent_form"] = loser_recent_form_list
df["winner_recent_matches"] = winner_recent_matches_list
df["loser_recent_matches"] = loser_recent_matches_list
df["winner_surface_relevance"] = winner_surface_relevance_list
df["loser_surface_relevance"] = loser_surface_relevance_list

df.to_csv(INTERIM_DIR / "atp_all_matches_with_features.csv", index = False)
print(f"Salvato! Shape: {df.shape}")
print(df[["winner_recent_form", "loser_recent_form", "winner_recent_matches"]].describe())


Salvato! Shape: (941649, 18)
       winner_recent_form  loser_recent_form  winner_recent_matches
count       871172.000000      815093.000000          941649.000000
mean             0.581334           0.541588               6.965293
std              0.250160           0.277085               4.926407
min              0.000000           0.000000               0.000000
25%              0.472257           0.393703               3.000000
50%              0.617017           0.575442               6.000000
75%              0.748454           0.720276              10.000000
max              1.000000           1.000000              43.000000
